In [ ]:
# --- Importation des bibliothèques nécessaires ---
# Spark
spark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
2,application_1742754072963_0003,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# Bibliothèques générales
import pandas as pd
from PIL import Image
import numpy as np
import io

# Bibliothèques TensorFlow et Keras
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras import Model

# Bibliothèques PySpark
from pyspark.sql.functions import col, pandas_udf, PandasUDFType, element_at, split
from pyspark.ml.feature import PCA
from pyspark.ml.linalg import Vectors, VectorUDT
import pyspark.sql.functions as F

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# --- Définition des chemins S3 ---
PATH = 's3://p11-bucket-1742660397'
PATH_Data = PATH+'/Test1'
PATH_Result = PATH+'/Results'

# Affichage des chemins pour vérification
print('PATH:        '+\
      PATH+'\nPATH_Data:   '+\
      PATH_Data+'\nPATH_Result: '+PATH_Result)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

PATH:        s3://p11-bucket-1742660397
PATH_Data:   s3://p11-bucket-1742660397/Test1
PATH_Result: s3://p11-bucket-1742660397/Results

In [ ]:
# --- Chargement des images au format binaire ---
#   - pathGlobFilter *.jpg : on ne charge que les fichiers .jpg
#   - recursiveFileLookup : on lit récursivement les répertoires
images = (spark.read.format("binaryFile")
          .option("pathGlobFilter", "*.jpg")
          .option("recursiveFileLookup", "true")
          .load(PATH_Data)
         )

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# Affichage de quelques lignes pour s'assurer que tout est correct
images.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+-------------------+------+--------------------+
|                path|   modificationTime|length|             content|
+--------------------+-------------------+------+--------------------+
|s3://p11-bucket-1...|2025-03-22 16:32:45| 10412|[FF D8 FF E0 00 1...|
|s3://p11-bucket-1...|2025-03-22 16:32:45| 10224|[FF D8 FF E0 00 1...|
|s3://p11-bucket-1...|2025-03-22 16:32:45|  9931|[FF D8 FF E0 00 1...|
|s3://p11-bucket-1...|2025-03-22 16:32:45|  9873|[FF D8 FF E0 00 1...|
|s3://p11-bucket-1...|2025-03-22 16:32:45|  9813|[FF D8 FF E0 00 1...|
+--------------------+-------------------+------+--------------------+
only showing top 5 rows

In [ ]:
# --- Extraction du label depuis le chemin ---
#   - On utilise la fonction split sur la colonne 'path' (séparateur '/'), puis on récupère
#     l'élément -2 (avant-dernier) pour extraire le label du dossier
images = images.withColumn('label', element_at(split(images['path'], '/'),-2))

# Affichage du schéma et d'un aperçu des colonnes path et label
print(images.printSchema())
print(images.select('path','label').show(5,False))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)
 |-- label: string (nullable = true)

None
+-------------------------------------------------------------+-------------+
|path                                                         |label        |
+-------------------------------------------------------------+-------------+
|s3://p11-bucket-1742660397/Test1/Blackberrie 2/r0_203_100.jpg|Blackberrie 2|
|s3://p11-bucket-1742660397/Test1/Blackberrie 2/r0_207_100.jpg|Blackberrie 2|
|s3://p11-bucket-1742660397/Test1/Blackberrie 2/r0_199_100.jpg|Blackberrie 2|
|s3://p11-bucket-1742660397/Test1/Blackberrie 2/r0_195_100.jpg|Blackberrie 2|
|s3://p11-bucket-1742660397/Test1/Blackberrie 2/r0_191_100.jpg|Blackberrie 2|
+-------------------------------------------------------------+-------------+
only showing top 5 rows

None

In [ ]:
# --- Chargement du modèle MobileNetV2 ---
#   - include_top=True : on conserve la dernière couche dense, initialement pour classifier 1000 classes
#   - On l'utilisera pourtant seulement jusqu'à l'avant-dernier layer pour de la featurisation
model = MobileNetV2(weights='imagenet',
                    include_top=True,
                    input_shape=(224, 224, 3))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

14536120/14536120 [==============================] - 1s 0us/step

In [ ]:
# Création d'un "nouveau" modèle en ne prenant que l'avant-dernière couche
new_model = Model(inputs=model.input, outputs=model.layers[-2].output)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# Broadcast des poids sur les workers Spark
brodcast_weights = sc.broadcast(new_model.get_weights())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# Résumé du modèle (utile pour comprendre la structure des couches)
new_model.summary()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 Conv1 (Conv2D)                 (None, 112, 112, 32  864         ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 bn_Conv1 (BatchNormalization)  (None, 112, 112, 32  128         ['Conv1[0][0]']                  
                                )                                                             

In [ ]:
# --- Définition d'une fonction qui renvoie un modèle MobileNetV2 ---
def model_fn():
    """
    Construit et renvoie un modèle MobileNetV2 (pré-entraîné sur ImageNet),
    tronqué à l'avant-dernière couche, et lui applique les poids broadcastés.

    Returns:
        new_model (tf.keras.Model): Modèle MobileNetV2 tronqué (featurisation).
    """
    # Chargement du modèle complet
    model = MobileNetV2(weights='imagenet',
                        include_top=True,
                        input_shape=(224, 224, 3))
    # On gel (freeze) toutes les couches pour la featurisation
    for layer in model.layers:
        layer.trainable = False
    # On construit le sous-modèle sans la dernière couche de classification
    new_model = Model(inputs=model.input,
                  outputs=model.layers[-2].output)
    # On applique les poids préalablement broadcastés (pour éviter de les recharger à chaque batch)
    new_model.set_weights(brodcast_weights.value)
    return new_model

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
def preprocess(content):
    """
    Prétraitement d'une image brute (bytes) pour la featurisation :
    - Conversion en image PIL
    - Redimensionnement en 224 x 224
    - Conversion en tableau numpy
    - Application du preprocess_input spécifique à MobileNetV2
    """
    img = Image.open(io.BytesIO(content)).resize([224, 224])
    arr = img_to_array(img)
    return preprocess_input(arr)

def featurize_series(model, content_series):
    """
    Featurise un batch d'images contenu dans une pd.Series de bytes via le modèle donné.
    Args:
        model (tf.keras.Model): le modèle de featurisation (MobileNetV2 tronqué).
        content_series (pd.Series): contient les bytes des images.

    Returns:
        pd.Series: chaque élément est un vecteur de features (extraits par le modèle).
    """
    # Prétraitement des images
    input_data = np.stack(content_series.map(preprocess))

    # Prédiction (featurisation) du batch
    preds = model.predict(input_data)

    # Flatten des features pour avoir un vecteur 1D par image
    output = [p.flatten() for p in preds]
    return pd.Series(output)

@pandas_udf('array<float>', PandasUDFType.SCALAR_ITER)
def featurize_udf(content_series_iter):
    """
    UDF Pandas (Scalar Iterator) permettant de featuriser des images par batch.

    Args:
        content_series_iter (iterator): itérateur de pd.Series (chacune correspondant à un batch).

    Yields:
        pd.Series: vecteurs de features extraits pour chaque batch.
    """
    # On charge le modèle une seule fois pour tous les batches (amortissement du coût)
    model = model_fn()

    # Pour chaque batch, on applique la featurisation et on renvoie une pd.Series de vecteurs
    for content_series in content_series_iter:
        yield featurize_series(model, content_series)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

/mnt/yarn/usercache/livy/appcache/application_1742754072963_0003/container_1742754072963_0003_01_000001/pyspark.zip/pyspark/sql/pandas/functions.py:403: UserWarning: In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.

In [ ]:
# --- Application de la featurisation à l'ensemble des images ---
#   - On reparitionne pour distribuer la charge
features_df = images.repartition(20).select(col("path"),
                                            col("label"),
                                            featurize_udf("content").alias("features") # On applique la UDF
                                           )

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# --- Conversion de l'array<float> en VectorUDT (format ML de Spark) ---
df_with_vectors = features_df.select(
    "path",
    "label",
    F.udf(lambda x: Vectors.dense(x), VectorUDT())("features").alias("features_vector")
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# Taille de la couche de features MobileNetV2 (include_top=True => 1280)
feature_dim = 1280

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# --- PCA "temporaire" pour analyser la variance expliquée et choisir le k optimal ---
pca_temp = PCA(k=feature_dim, inputCol="features_vector", outputCol="features_temp")
pca_temp_model = pca_temp.fit(df_with_vectors)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# Récupération du pourcentage de variance expliqué par composante
explained_variance = pca_temp_model.explainedVariance.toArray()
# Calcul de la somme cumulée
cumsum_var = np.cumsum(explained_variance)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# On fixe un seuil de variance expliquée (90 % ou 95 % dans beaucoup de cas)
threshold = 0.90
# On trouve la plus petite dimension k telle que la variance cumulée dépasse le seuil
optimal_k = np.searchsorted(cumsum_var, threshold) + 1  # +1 car recherche en indice 0-based

# Affichage des premières valeurs pour diagnostic
print("==================================================")
print("Variance expliquée (premiers éléments) :", explained_variance[:10], "...")
print("Variance cumulée (premiers éléments)   :", cumsum_var[:10], "...")
print(f"Nombre optimal de composantes pour atteindre {int(threshold*100)}% de variance =", optimal_k)
print("==================================================")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Variance expliqu?e (premiers ?l?ments) : [0.09500267 0.07452281 0.06089583 0.04685572 0.03660199 0.02939862
 0.02686523 0.02328727 0.02056316 0.0190709 ] ...
Variance cumul?e (premiers ?l?ments)   : [0.09500267 0.16952548 0.23042131 0.27727704 0.31387902 0.34327764
 0.37014288 0.39343015 0.41399331 0.43306421] ...
Nombre optimal de composantes pour atteindre 90% de variance = 207

In [ ]:
# --- PCA "final" avec le k optimal ---
pca_final = PCA(k=optimal_k, inputCol="features_vector", outputCol="features_pca")
pca_final_model = pca_final.fit(df_with_vectors)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# Transformation des features par la PCA réduite
df_pca_final = pca_final_model.transform(df_with_vectors)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# Vérification de l'emplacement de sortie
print(PATH_Result)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

s3://p11-bucket-1742660397/Results

In [ ]:
# Chemin de sortie final pour stocker les vecteurs PCA
output_path = PATH_Result + "/pca_features"

# Écriture au format parquet sur S3 (en overwrite)
df_pca_final.select("path", "label", "features_pca").write.mode("overwrite").parquet(output_path)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# --- Vérification en local côté driver ---
#   - Chargement du parquet final avec pandas (lecture pyarrow)
df = pd.read_parquet(output_path, engine='pyarrow')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# Affichage des premières lignes pour contrôle
df.head()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

                                                path  ...                                       features_pca
0  s3://p11-bucket-1742660397/Test1/Apple Rotten ...  ...  {'type': 1, 'size': None, 'indices': None, 'va...
1  s3://p11-bucket-1742660397/Test1/Cabbage red 1...  ...  {'type': 1, 'size': None, 'indices': None, 'va...
2  s3://p11-bucket-1742660397/Test1/Blackberrie n...  ...  {'type': 1, 'size': None, 'indices': None, 'va...
3  s3://p11-bucket-1742660397/Test1/Cactus fruit ...  ...  {'type': 1, 'size': None, 'indices': None, 'va...
4  s3://p11-bucket-1742660397/Test1/Cactus fruit ...  ...  {'type': 1, 'size': None, 'indices': None, 'va...

[5 rows x 3 columns]

In [ ]:
# Affichage d'informations sur les dimensions finales
print("Dimensions du vecteur PCA :", len(df.loc[0, 'features_pca']))
print("Taille du jeu de données  :", df.shape)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Dimensions du vecteur PCA : 4